# 미니 프로젝트 템플릿 ① — 이미지 분류 (17_project_image_template)

9주차 `14_image_project` 를 **내 프로젝트용으로 넓힌 템플릿**입니다. 빈칸은 없습니다.
**셀 0 의 설정**과 **실험 슬롯(셀 5~7)의 인자**만 바꿔 가며 실험합니다.

```
  셀 0  설정 (데이터 폴더 · 사진 크기 · 배치)          ← 내 프로젝트에 맞게 고친다
  셀 1  데이터 불러오기 + 클래스별 장수 점검
  셀 2  사진 확인
  셀 3  도구 함수 (모델 만들기 · 학습 · 실험 기록)      ← 고치지 않는다
  셀 4  E0 기준선 — ResNet18 백본 동결
  셀 5~7  E1·E2·E3 개선 실험 슬롯                     ← 인자만 바꾼다
  셀 8  실험 기록표 (results/experiments.csv)
  셀 9  가장 좋은 실험 평가 — 혼동행렬 · 클래스별 성적
  셀 10 틀린 사진 보기 (오분류 분석)
  셀 11 내 사진 한 장 예측
  셀 12 보고서용 요약
```

### 준비

- `mydata/train/<클래스>/` · `mydata/val/<클래스>/` 구조 (터미널에서 `python prepare_mydata.py` 로 점검)
- 같은 폴더에 `sample_data.py`, `prepare_mydata.py` — `mydata` 가 없으면 연습용 데이터(비행기·자동차·고양이)로 실행됩니다
- 실험할 때마다 결과가 **`results/experiments.csv`** 에 한 줄씩 쌓입니다. 보고서의 실험 비교표는 이 파일에서 옮깁니다.

### 실험 기록표의 열

| 열 | 뜻 |
|---|---|
| `exp_id` | 실험 번호 (E0 = 기준선, E1·E2·E3 … = 개선 실험) |
| `timestamp` | 실행한 날짜·시각 |
| `task` | `image` 또는 `text` |
| `method` | 모델 방식 (예: `resnet_frozen`, `finetune`) |
| `change` | **기준선에서 무엇을 바꿨는지** 한 줄 (내가 적는 칸) |
| `setting` | 그 밖의 설정 (증강 여부, 문장 최대 길이 등) |
| `epochs / lr / batch_size` | 학습 설정 |
| `trainable_params` | 학습한 파라미터 수 |
| `best_val_acc / best_epoch` | 최고 검증 정확도와 그때의 epoch |
| `final_train_loss / final_val_loss` | 마지막 epoch 의 훈련·검증 손실 (차이가 크면 과적합 의심) |
| `train_sec` | 학습 시간(초) |
| `model_path` | 최고 시점 가중치 파일 |

In [ ]:
# 셀 0 — ★ 설정 : 내 프로젝트에 맞게 이 셀만 고친다
PROJECT = "내 이미지 분류 프로젝트"     # 보고서 제목
DATA    = "mydata"                     # train/ val/ 클래스 폴더가 들어 있는 폴더
IMG     = 128                          # 사진 크기 (128 권장. GPU 가 넉넉하면 224)
BATCH   = 16                           # GPU 메모리가 부족하다는 오류가 나면 8
SEED    = 42                           # 같은 결과를 다시 얻기 위한 값 (바꾸지 않는다)

In [ ]:
# 셀 1 — 준비와 데이터 불러오기
import os, csv, time, random, datetime
import torch, torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models
from torchvision.datasets import ImageFolder
from torchvision.transforms import v2
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False
os.makedirs("models", exist_ok=True); os.makedirs("results", exist_ok=True)

def set_seed(s=SEED):
    random.seed(s); torch.manual_seed(s)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("장치 :", device)

if not os.path.isdir(DATA):
    print(f"'{DATA}' 폴더가 없어 연습용 데이터로 실행합니다.")
    from sample_data import make_sample
    DATA = make_sample()

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)     # ImageNet 표준값 (전이학습용)
AUG_TF = v2.Compose([v2.RandomResizedCrop(IMG, scale=(0.8, 1.0)), v2.RandomHorizontalFlip(),
                     v2.ToImage(), v2.ToDtype(torch.float32, scale=True), v2.Normalize(MEAN, STD)])
PLAIN_TF = v2.Compose([v2.Resize((IMG, IMG)),
                       v2.ToImage(), v2.ToDtype(torch.float32, scale=True), v2.Normalize(MEAN, STD)])

val_set = ImageFolder(f"{DATA}/val", transform=PLAIN_TF)
val_loader = DataLoader(val_set, batch_size=32)
CLASSES = val_set.classes
NUM_CLASSES = len(CLASSES)

def make_train_loader(augment=True):
    ds = ImageFolder(f"{DATA}/train", transform=AUG_TF if augment else PLAIN_TF)
    assert ds.classes == CLASSES, "train 과 val 의 클래스 폴더 이름이 다릅니다"
    return DataLoader(ds, batch_size=BATCH, shuffle=True)

train_raw = ImageFolder(f"{DATA}/train")
count = {c: 0 for c in CLASSES}
for _, y in train_raw.samples: count[CLASSES[y]] += 1
vcount = {c: 0 for c in CLASSES}
for _, y in val_set.samples: vcount[CLASSES[y]] += 1

print("데이터 폴더 :", DATA)
print(f"클래스 {NUM_CLASSES}개")
for c in CLASSES:
    print(f"   {c:<10} 훈련 {count[c]:4d}장 / 검증 {vcount[c]:3d}장")
if max(count.values()) > 3 * min(count.values()):
    print("⚠️ 클래스별 장수 차이가 큽니다 → 정확도만 보지 말고 클래스별 재현율도 보세요")
if min(vcount.values()) < 5:
    print("⚠️ 검증 사진이 5장 미만인 클래스가 있습니다 → 정확도가 크게 흔들립니다")

In [ ]:
# 셀 2 — 사진 확인 (증강 적용된 훈련 배치)
xb, yb = next(iter(make_train_loader(augment=True)))
def show(x):
    return (x.permute(1, 2, 0) * torch.tensor(STD) + torch.tensor(MEAN)).clamp(0, 1)
n = min(8, len(xb))
fig, ax = plt.subplots(1, n, figsize=(2 * n, 2.4))
for i in range(n):
    ax[i].imshow(show(xb[i])); ax[i].set_title(CLASSES[yb[i]]); ax[i].axis("off")
plt.tight_layout(); plt.show()

---

## 셀 3 — 도구 함수 (고치지 않습니다)

| 함수 | 하는 일 |
|---|---|
| `build_model(method)` | 모델 만들기 — `"resnet_frozen"` (ResNet18 백본 동결) · `"resnet_finetune"` (전체 미세조정) · `"cnn"` (처음부터 만든 CNN) |
| `run_experiment(...)` | 학습 → 최고 시점 저장 → 곡선 그리기 → **실험 기록표에 한 줄 추가** |
| `show_log()` | 지금까지의 실험 기록표 보기 |

In [ ]:
# 셀 3 — 도구 함수
def block(i, o):
    return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2))

class MyCNN(nn.Module):                       # 9주차에 만든 CNN 과 같은 구조
    def __init__(self, n):
        super().__init__()
        self.features = nn.Sequential(block(3, 32), block(32, 64), block(64, 128))
        self.classifier = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(128, n))
    def forward(self, x):
        return self.classifier(self.features(x))

def build_model(method):
    set_seed()
    if method == "cnn":
        m = MyCNN(NUM_CLASSES)
    elif method in ("resnet_frozen", "resnet_finetune"):
        m = models.resnet18(weights="DEFAULT")
        if method == "resnet_frozen":
            for p in m.parameters():
                p.requires_grad = False          # 백본 동결
        m.fc = nn.Linear(512, NUM_CLASSES)      # 헤드 교체 (새 층은 학습 가능)
    else:
        raise ValueError(f"모르는 방식입니다 : {method}")
    return m.to(device)

loss_fn = nn.CrossEntropyLoss()

@torch.no_grad()
def evaluate(model, loader=None):
    loader = loader or val_loader
    model.eval(); total = correct = n = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        out = model(xb)
        total += loss_fn(out, yb).item() * len(yb)
        correct += (out.argmax(1) == yb).sum().item(); n += len(yb)
    return total / n, correct / n

LOG_PATH = "results/experiments.csv"
LOG_COLS = ['exp_id', 'timestamp', 'task', 'method', 'change', 'setting', 'epochs', 'lr', 'batch_size', 'trainable_params', 'best_val_acc', 'best_epoch', 'final_train_loss', 'final_val_loss', 'train_sec', 'model_path']

def log_experiment(row):
    """실험 한 번의 결과를 results/experiments.csv 에 한 줄 덧붙인다 (지우지 않고 계속 쌓인다)."""
    new = not os.path.exists(LOG_PATH)
    with open(LOG_PATH, "a", encoding="utf-8-sig", newline="") as f:
        w = csv.DictWriter(f, fieldnames=LOG_COLS)
        if new:
            w.writeheader()
        w.writerow({k: row.get(k, "") for k in LOG_COLS})

def read_log():
    if not os.path.exists(LOG_PATH):
        return []
    with open(LOG_PATH, encoding="utf-8-sig") as f:
        return list(csv.DictReader(f))

def show_log():
    rows = read_log()
    if not rows:
        print("아직 기록된 실험이 없습니다."); return
    print(f"{'번호':<5}{'방식':<17}{'최고 검증 정확도':>16}{'epoch':>7}{'학습 파라미터':>15}{'시간(초)':>10}   바꾼 것")
    print("-" * 100)
    for r in rows:
        print(f"{r['exp_id']:<5}{r['method']:<17}{float(r['best_val_acc'])*100:>15.1f}%{r['best_epoch']:>7}"
              f"{int(r['trainable_params']):>15,}{float(r['train_sec']):>10.1f}   {r['change']}")
    best = max(rows, key=lambda r: float(r["best_val_acc"]))
    print(f"\n가장 좋은 실험 : {best['exp_id']} ({best['change']}) — {float(best['best_val_acc'])*100:.1f}%")

def plot_history(h, title):
    ep = range(1, len(h["train_loss"]) + 1)
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
    ax[0].plot(ep, h["train_loss"], "o-", label="훈련 손실"); ax[0].plot(ep, h["val_loss"], "o-", label="검증 손실")
    ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].grid(alpha=.3); ax[0].set_title(f"{title} — 손실")
    ax[1].plot(ep, [a * 100 for a in h["val_acc"]], "o-", color="tab:green")
    ax[1].set_xlabel("epoch"); ax[1].set_ylabel("%"); ax[1].grid(alpha=.3); ax[1].set_title(f"{title} — 검증 정확도")
    plt.tight_layout(); plt.savefig(f"results/{title.split()[0]}_curve.png", dpi=110); plt.show()

def run_experiment(exp_id, change, method="resnet_frozen", epochs=8, lr=1e-3, augment=True):
    """실험 한 번 = 학습 + 최고 시점 저장 + 곡선 + 기록표 한 줄"""
    model = build_model(method)
    train_loader = make_train_loader(augment)
    set_seed()
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    h = {"train_loss": [], "val_loss": [], "val_acc": []}
    best, best_ep, path, t0 = -1.0, 0, f"models/{exp_id}.pt", time.time()
    for ep in range(1, epochs + 1):
        model.train(); run = n = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            loss = loss_fn(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
            run += loss.item() * len(yb); n += len(yb)
        vl, va = evaluate(model)
        h["train_loss"].append(run / n); h["val_loss"].append(vl); h["val_acc"].append(va)
        mark = ""
        if va > best:
            best, best_ep = va, ep
            torch.save(model.state_dict(), path); mark = "  ← 최고, 저장"
        print(f"[{exp_id}] epoch {ep:2d} | 훈련 손실 {run/n:.4f} | 검증 손실 {vl:.4f} | 검증 정확도 {va*100:5.1f}%{mark}")
    sec = time.time() - t0
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"[{exp_id}] 최고 검증 정확도 {best*100:.1f}% (epoch {best_ep}) | 학습 파라미터 {trainable:,} | {sec:.1f}초")
    log_experiment(dict(exp_id=exp_id, timestamp=datetime.datetime.now().strftime("%Y-%m-%d %H:%M"),
                        task="image", method=method, change=change, setting=f"augment={augment}, img={IMG}",
                        epochs=epochs, lr=lr, batch_size=BATCH, trainable_params=trainable,
                        best_val_acc=round(best, 4), best_epoch=best_ep,
                        final_train_loss=round(h["train_loss"][-1], 4), final_val_loss=round(h["val_loss"][-1], 4),
                        train_sec=round(sec, 1), model_path=path))
    plot_history(h, f"{exp_id} {change}")
    return h

print("도구 준비 완료 : build_model / run_experiment / show_log")

---

## 셀 4 — E0 기준선

**기준선(baseline)** = 앞으로의 실험을 비교할 **기준점**입니다. 9주차에 가장 좋았던 방식(ResNet18 백본 동결)을 그대로 씁니다.
개선 실험은 모두 **"기준선보다 나아졌나?"** 로 판단합니다.

In [ ]:
# 셀 4 — E0 기준선 : ResNet18 백본 동결 (9주차 방식)
h0 = run_experiment("E0", "기준선 (ResNet18 동결)", method="resnet_frozen", epochs=8, lr=1e-3)

---

## 셀 5~7 — 개선 실험 슬롯

**한 번에 하나만 바꿉니다.** 두 가지를 동시에 바꾸면 무엇 때문에 좋아졌는지 알 수 없습니다.
`change` 에는 **무엇을 바꿨는지** 한 줄로 적습니다 — 실험 기록표와 보고서에 그대로 남습니다.

| 바꿔 볼 것 | 인자 | 예시 |
|---|---|---|
| 모델 방식 | `method` | `"resnet_finetune"` (전체 미세조정) · `"cnn"` (처음부터) |
| 학습률 | `lr` | 동결 `1e-3` → `3e-3` / 미세조정은 작게 `1e-4` |
| epoch 수 | `epochs` | 8 → 15 (곡선이 아직 내려가고 있다면) |
| 증강 | `augment` | `True` → `False` (증강의 효과 확인) |
| 사진 크기 | 셀 0 `IMG` | 128 → 224 (셀 0 부터 다시 실행) |
| 데이터 | 폴더 | 사진 추가 · 잘못 분류된 사진 정리 (가장 효과가 큰 경우가 많다) |

아래 세 슬롯은 **예시로 채워 둔 값**입니다. 내 프로젝트에 맞게 바꿔 쓰세요. 슬롯이 더 필요하면 셀을 복사해 `E4`, `E5` … 로.

In [ ]:
# 셀 5 — E1 : (예시) 전체 미세조정 — 백본까지 학습, 학습률은 작게
h1 = run_experiment("E1", "전체 미세조정 (lr 1e-4)", method="resnet_finetune", epochs=5, lr=1e-4)

In [ ]:
# 셀 6 — E2 : (예시) 증강을 끄면 어떻게 되나
h2 = run_experiment("E2", "증강 끄기", method="resnet_frozen", epochs=8, lr=1e-3, augment=False)

In [ ]:
# 셀 7 — E3 : (예시) 처음부터 만든 CNN 과 비교
h3 = run_experiment("E3", "처음부터 CNN", method="cnn", epochs=15, lr=1e-3)

---

## 셀 8 — 실험 기록표

`results/experiments.csv` 는 **엑셀로도 열립니다.** 보고서의 실험 비교표는 이 표를 옮겨 적습니다.
같은 번호로 다시 실행하면 **새 줄이 추가**됩니다 (기록이 지워지지 않는 것이 정상입니다).

In [ ]:
# 셀 8 — 실험 기록표 보기
show_log()

---

## 셀 9 — 가장 좋은 실험 평가

`BEST_ID` 를 비워 두면 **검증 정확도가 가장 높은 실험**을 자동으로 고릅니다. 특정 실험을 보고 싶으면 `"E1"` 처럼 적습니다.

In [ ]:
# 셀 9 — 가장 좋은 실험 : 혼동행렬 + 클래스별 성적
BEST_ID = ""                                   # 비우면 자동 선택

rows = read_log()
pick = [r for r in rows if r["exp_id"] == BEST_ID] if BEST_ID else rows
best_row = max(pick, key=lambda r: float(r["best_val_acc"]))
best_model = build_model(best_row["method"])
best_model.load_state_dict(torch.load(best_row["model_path"], map_location=device))
print(f"평가할 실험 : {best_row['exp_id']} ({best_row['change']})")

@torch.no_grad()
def predict_all(model):
    model.eval(); ys, ps = [], []
    for xb, yb in val_loader:
        ps += model(xb.to(device)).argmax(1).cpu().tolist(); ys += yb.tolist()
    return ys, ps

ys, ps = predict_all(best_model)
fig, ax = plt.subplots(figsize=(1.3 * NUM_CLASSES + 2.5, 1.3 * NUM_CLASSES + 2))
ConfusionMatrixDisplay(confusion_matrix(ys, ps), display_labels=CLASSES).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"{best_row['exp_id']} 혼동행렬"); ax.set_xlabel("모델의 예측"); ax.set_ylabel("정답")
plt.tight_layout(); plt.savefig("results/confusion_matrix.png", dpi=120); plt.show()
print(classification_report(ys, ps, target_names=CLASSES, digits=3, zero_division=0))

> **읽는 법** — 대각선 = 맞힌 것, 대각선 밖 = 헷갈린 것.
> **precision** 헛짚지 않았나 · **recall** 놓치지 않았나 · **f1** 둘의 균형. 보고서에는 **가장 많이 헷갈린 클래스 쌍**과 그 이유를 적습니다.

In [ ]:
# 셀 10 — 틀린 사진 보기 : 왜 틀렸을지 추측해 보는 것이 결과 해석의 시작
wrong = [(i, y, p) for i, (y, p) in enumerate(zip(ys, ps)) if y != p]
print(f"검증 {len(ys)}장 중 틀린 사진 {len(wrong)}장")
if wrong:
    from PIL import Image
    k = min(8, len(wrong))
    fig, ax = plt.subplots(1, k, figsize=(2.2 * k, 2.8), squeeze=False)
    for a, (i, y, p) in zip(ax[0], wrong[:k]):
        a.imshow(Image.open(val_set.samples[i][0]).convert("RGB")); a.axis("off")
        a.set_title(f"정답 {CLASSES[y]}\n예측 {CLASSES[p]}", fontsize=10, color="crimson")
    plt.tight_layout(); plt.savefig("results/wrong_examples.png", dpi=110); plt.show()

In [ ]:
# 셀 11 — 내 사진 한 장 예측 (학습에 쓰지 않은 사진)
from PIL import Image
MY_PHOTO = ""                                   # 예) "test_photo.jpg"  — 비우면 검증 사진 첫 장

@torch.no_grad()
def predict_one(model, path):
    img = Image.open(path).convert("RGB")
    model.eval()
    prob = torch.softmax(model(PLAIN_TF(img).unsqueeze(0).to(device)), dim=1)[0].cpu()
    plt.figure(figsize=(3, 3)); plt.imshow(img); plt.axis("off")
    plt.title(f"{CLASSES[prob.argmax()]} ({prob.max()*100:.1f}%)"); plt.show()
    for c, p in sorted(zip(CLASSES, prob.tolist()), key=lambda t: -t[1]):
        print(f"  {c:<10} {p*100:5.1f}%")

path = MY_PHOTO or val_set.samples[0][0]
print("사진 :", os.path.normpath(path))
predict_one(best_model, path)

In [ ]:
# 셀 12 — 보고서용 요약 (그대로 복사해 보고서 초안에 붙인다)
rows = read_log()
base = next((r for r in rows if r["exp_id"] == "E0"), rows[0])
print(f"■ 프로젝트 : {PROJECT}")
print(f"■ 데이터   : 클래스 {NUM_CLASSES}개 {CLASSES} · 훈련 {sum(count.values())}장 · 검증 {sum(vcount.values())}장")
print(f"■ 기준선   : {base['exp_id']} {base['change']} → 검증 정확도 {float(base['best_val_acc'])*100:.1f}%")
print(f"■ 최고     : {best_row['exp_id']} {best_row['change']} → 검증 정확도 {float(best_row['best_val_acc'])*100:.1f}%"
      f" (기준선 대비 {(float(best_row['best_val_acc'])-float(base['best_val_acc']))*100:+.1f}%p)")
print(f"■ 실험 수  : {len(rows)}회  (results/experiments.csv)")
print("■ 그림     : results/confusion_matrix.png · results/wrong_examples.png · results/E*_curve.png")